In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install dagshub mlflow -q

# Model Inference

## Setup And Imports

In [4]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='fraud-detection', mlflow=True)

print('done')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=5221883d-f1f1-4a01-ab90-463273e353ee&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5f6b7e83dec64b9d918eec89572d0c46250c52ba626a7bdc03c639653e151817




Accessing as LukaBatilashvili07

Initialized MLflow to track repo "LukaBatilashvili07/fraud-detection"

Repository LukaBatilashvili07/fraud-detection initialized!

done


## Data Loading

In [5]:
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

test = test_transaction.merge(test_identity, on='TransactionID', how='left')
test_ids = test['TransactionID']

print('Test shape:', test.shape)
test.head()

Test shape: (506691, 433)


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Load Model

In [9]:
model = mlflow.sklearn.load_model("models:/best_fraud_model/1")
print(model)

Pipeline(steps=[('clf',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=1.0, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='auc',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,


## Preprocessing

In [12]:
X_test = test.drop(['TransactionID'], axis=1)

null_ratio = X_test.isnull().mean()
drop_cols = null_ratio[null_ratio > 0.5].index
X_test = X_test.drop(columns=drop_cols, errors='ignore')

cat_cols = X_test.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in cat_cols:
    X_test[col] = X_test[col].astype(str)
    le.fit(X_test[col])
    X_test[col] = le.transform(X_test[col])

X_test = X_test.fillna(X_test.median())

X_test['amt_log'] = np.log1p(X_test['TransactionAmt'])
X_test['amt_cents'] = X_test['TransactionAmt'] % 1

selected_cols = ['ProductCD', 'card3', 'V15', 'V16', 'V17', 'V18', 'V21', 'V22', 'V23', 
                 'V31', 'V32', 'V33', 'V34', 'V37', 'V38', 'V39', 'V40', 'V42', 'V43', 
                 'V44', 'V45', 'V47', 'V50', 'V51', 'V52', 'V57', 'V58', 'V59', 'V60', 
                 'V63', 'V64', 'V71', 'V72', 'V73', 'V74', 'V77', 'V78', 'V79', 'V80', 
                 'V81', 'V84', 'V85', 'V86', 'V87', 'V92', 'V93', 'V94', 'V123', 'V302', 'V304']

X_test = X_test[selected_cols]

print('Shape:', X_test.shape)

Shape: (506691, 50)


## Prediction

In [13]:
preds = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': preds
})

submission.to_csv('submission.csv', index=False)
print(submission.head())

   TransactionID   isFraud
0        3663549  0.017100
1        3663550  0.017100
2        3663551  0.025721
3        3663552  0.014354
4        3663553  0.017100
